In [ ]:
# Builds the BF16 vs INT4 figures and 18-view GIFs from the renders. No GPU needed.
# One PNG per prompt: BF16 and INT4 full meshes on top, their parts below.
# Set a prompt's best view in VIEW_OVERRIDE, e.g. {"020": 3}, and rerun.
# Pass 1: all prompts from sh_render_out_v2, to choose. Pass 2: RENDER_DIR = sh_render_out_hd,
# FIG_DIR = sh_figures_hd and ONLY = the chosen prompts.
import json
import os
import textwrap
import zipfile

import numpy as np
from google.colab import drive
from PIL import Image, ImageDraw, ImageFont

ZIP_PATH = "/content/drive/MyDrive/CubePart_nums/visual_assets_36prompts.zip"
RENDER_DIR = "/content/drive/MyDrive/sh_render_out_v2"
FIG_DIR = "/content/drive/MyDrive/sh_figures"

ONLY = []           # e.g. ["020", "130"]; empty means all 36
DEFAULT_VIEW = 0
VIEW_OVERRIDE = {}
HALF_W = 1400       # width of the BF16 half and of the INT4 half
FULL_H = 1100       # full mesh panel height
GAP = 30
MARGIN = 0.06       # empty border inside each panel
GIF_SIZE = 512
GIF_MS = 300

drive.mount("/content/drive")
os.makedirs(FIG_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    manifest = json.loads(z.read("visual_assets/manifest.json"))

font = ImageFont.load_default(size=40)
small = ImageFont.load_default(size=32)
missing = []


def stem(path):
    return os.path.splitext(os.path.basename(path))[0]


def view_path(name, view):
    return os.path.join(RENDER_DIR, name, f"view_{view:02d}.png")


def panel(name, view, w, h):
    """Crops the render to the object and scales it uniformly, so nothing is distorted."""
    out = Image.new("RGB", (w, h), "black")
    path = view_path(name, view) if name else None
    if path is None or not os.path.exists(path):
        missing.append(name or "part with no mesh")
        ImageDraw.Draw(out).text((w // 2, h // 2), "no mesh", fill="white", font=small, anchor="mm")
        return out
    img = Image.open(path).convert("RGB")
    box = img.convert("L").getbbox()
    if box:
        img = img.crop(box)
    scale = min(w * (1 - 2 * MARGIN) / img.width, h * (1 - 2 * MARGIN) / img.height)
    img = img.resize((max(1, round(img.width * scale)), max(1, round(img.height * scale))), Image.LANCZOS)
    out.paste(img, ((w - img.width) // 2, (h - img.height) // 2))
    return out


def part_names(parts, n):
    by_index = {p["part_index"]: stem(p["file"]) for p in parts}
    return [by_index.get(i) for i in range(n)]


def wrapped(text, width=70):
    return textwrap.wrap(text, width) or [""]


def build_figure(idx, entry):
    view = VIEW_OVERRIDE.get(idx, DEFAULT_VIEW)
    labels = entry["parts_list"]
    n = len(labels)
    part_w = (HALF_W - GAP * (n - 1)) // n
    prompt_lines = wrapped("Cube3D Prompt: " + entry["prompt"])
    list_lines = wrapped("CubePart list: " + ", ".join(labels))
    text_h = 52 * max(len(prompt_lines), len(list_lines))
    label_lines = [textwrap.wrap(l, max(8, part_w // 18)) for l in labels]
    label_h = 42 * max(len(l) for l in label_lines)
    width = 2 * HALF_W + 3 * GAP
    height = GAP + 60 + FULL_H + GAP // 2 + text_h + GAP // 2 + part_w + label_h + GAP

    fig = Image.new("RGB", (width, height), "white")
    d = ImageDraw.Draw(fig)
    for col, tag in enumerate(("bf16", "int4")):
        x = GAP + col * (HALF_W + GAP)
        d.text((x + HALF_W // 2, GAP), f"{tag.upper()} Cube3D + {tag.upper()} CubePart", fill="black", font=font, anchor="ma")
        y = GAP + 60
        fig.paste(panel(stem(entry[f"cube3d_{tag}"]), view, HALF_W, FULL_H), (x, y))
        y += FULL_H + GAP // 2
        for k, line in enumerate(prompt_lines if col == 0 else list_lines):
            d.text((x, y + 52 * k), line, fill="black", font=font)
        y += text_h + GAP // 2
        for i, name in enumerate(part_names(entry[f"cubepart_{tag}_parts"], n)):
            px = x + i * (part_w + GAP)
            fig.paste(panel(name, view, part_w, part_w), (px, y))
            for k, line in enumerate(label_lines[i]):
                d.text((px + part_w // 2, y + part_w + 6 + 42 * k), line, fill="black", font=small, anchor="ma")
    fig.save(os.path.join(FIG_DIR, f"{idx}_figure.png"))


def orbit_order(n_views=18, max_elev=0.4):
    """Near-level views sorted by azimuth about +y, so the GIF reads as a turntable.
    Views from far above or below (top, bottom) are left out because they make it jump."""
    i = np.arange(n_views) + 0.5
    phi = np.arccos(1.0 - 2.0 * i / n_views)
    theta = np.pi * (1.0 + 5.0 ** 0.5) * i
    x, y, z = np.sin(phi) * np.cos(theta), np.sin(phi) * np.sin(theta), np.cos(phi)
    level = np.flatnonzero(np.abs(y) < max_elev)
    return level[np.argsort(np.arctan2(x[level], z[level]))]


def build_gif(name):
    paths = [view_path(name, int(v)) for v in orbit_order()]
    if not all(os.path.exists(p) for p in paths):
        missing.append(name)
        return
    frames = [Image.open(p).convert("RGB").resize((GIF_SIZE, GIF_SIZE), Image.LANCZOS) for p in paths]
    frames[0].save(os.path.join(FIG_DIR, f"{name}.gif"), save_all=True, append_images=frames[1:],
                   duration=GIF_MS, loop=0)


assert sorted(orbit_order().tolist()) == [0, 1, 2, 6, 10, 14, 15, 17]
manifest = {k: v for k, v in manifest.items() if not ONLY or k in ONLY}
assert manifest, "ONLY matches no prompt"
for idx, entry in sorted(manifest.items()):
    build_figure(idx, entry)
    build_gif(stem(entry["cube3d_bf16"]))
    build_gif(stem(entry["cube3d_int4"]))
print(f"{len(manifest)} figures and {2 * len(manifest)} GIFs in {FIG_DIR}")
print(f"{len(missing)} panels with no render:", missing[:20])
